# CCE Thermal Simulator — Examples

This notebook demonstrates the physics-based synthetic data generator for cold chain equipment (CCE) monitoring. It replaces the old database-dependent `RecordSetFactory` with a self-contained simulation.

In [ ]:
import datetime as dt
import matplotlib.pyplot as plt
from utils.simulator import SimulatedRecordSet, default_config
from utils.device import MonitoringDeviceConfig, BaseRtmDevice

## 1. Low-level: SimulatedRecordSet

Generate a 24-hour batch directly from the simulator engine.

In [ ]:
# Mains-powered fridge at latitude 12 (tropical)
config = default_config(power_type="mains", latitude=12.0)
config.sample_interval = 900  # 10-minute samples
sample_days = 7

start = dt.datetime(2024, 6, 15, 0, 0, 0)

batch_size = int(86400 / config.sample_interval * sample_days)  # * sample 7 days of data at 10-minute intervals
rs = SimulatedRecordSet.generate(config, batch_size=batch_size, start_time=start, interval=config.sample_interval)
# Extract time series
times = [r['ABST'] for r in rs.records]
tvcs = [r['TVC'] for r in rs.records]
tambs = [r['TAMB'] for r in rs.records]
cmprs = [r['CMPR'] for r in rs.records]
dorvs = [r['DORV'] for r in rs.records]

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

axes[0].plot(times, tvcs, label='TVC (vaccine chamber)', color='blue')
axes[0].plot(times, tambs, label='TAMB (ambient)', color='orange', alpha=0.6)
axes[0].axhline(2, color='cyan', linestyle='--', alpha=0.5, label='Setpoint low (2°C)')
axes[0].axhline(8, color='red', linestyle='--', alpha=0.5, label='Setpoint high (8°C)')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend(loc='upper right')
axes[0].set_title(f'Mains-powered fridge — {sample_days} days')

axes[1].bar(times, cmprs, width=dt.timedelta(minutes=8), color='green', alpha=0.7)
axes[1].set_ylabel('Compressor runtime (s)')
axes[1].set_xlabel('Time')

axes[2].plot(times, dorvs, label='DORV (door open time)', color='magenta')
axes[2].set_ylabel('Door open time (seconds)')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

## 2. Solar direct drive (SDD) fridge

Solar fridges have a DC voltage bell curve during daylight and battery SOC tracking.

In [ ]:
# Solar fridge at latitude 12
solar_config = default_config(power_type="solar", latitude=12.0)
solar_config.sample_interval = 900
sample_days = 7

solar_batch_size = int(86400 / solar_config.sample_interval * sample_days)  # * sample 7 days of data at 10-minute intervals

rs_solar = SimulatedRecordSet.generate(solar_config, batch_size=solar_batch_size, start_time=start, interval=solar_config.sample_interval)

times_s = [r['ABST'] for r in rs_solar.records]
tvcs_s = [r['TVC'] for r in rs_solar.records]
dcsvs = [r['DCSV'] for r in rs_solar.records]
tambs_s = [r['TAMB'] for r in rs_solar.records]
cmprs_s = [r['CMPR'] for r in rs_solar.records]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(times_s, tvcs_s, label='TVC', color='blue')
axes[0].plot(times_s, tambs_s, label='TAMB', color='orange', alpha=0.6)
axes[0].axhline(2, color='cyan', linestyle='--', alpha=0.5)
axes[0].axhline(8, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('TVC (°C)')
axes[0].set_title('Solar direct drive fridge — 24 hours')
axes[0].legend()

axes[1].plot(times_s, dcsvs, label='DCSV (solar voltage)', color='goldenrod')
axes[1].set_ylabel('DC Voltage (V)')
axes[1].legend()

axes[2].plot(times_s, cmprs_s, label='CMPR (compressor run time)', color='green')
axes[2].set_ylabel('Compressor runtime (s)')
axes[2].set_xlabel('Time')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Fault injection

Inject a compressor failure starting 8 hours into the simulation to see TVC excursion.

In [ ]:
from utils.simulator.config import FaultType

fault_config = default_config(power_type="mains", latitude=12.0)
fault_config.sample_interval = 900
fault_config.fault.fault_type = FaultType.POWER_OUTAGE
fault_config.fault.fault_start_offset_s = 8 * 3600   # Fail at hour 8
fault_config.fault.fault_duration_s = 24 * 3600 * 10       # Lasts 6 hours

sample_days = 14
batch_size = int(86400 / fault_config.sample_interval * sample_days)  # * sample 7 days of data at 10-minute intervals
rs_fault = SimulatedRecordSet.generate(fault_config, batch_size=batch_size, start_time=start, interval=fault_config.sample_interval)

times_f = [r['ABST'] for r in rs_fault.records]
tvcs_f = [r['TVC'] for r in rs_fault.records]
cmprs_f = [r['CMPR'] for r in rs_fault.records]
tambs_f = [r['TAMB'] for r in rs_fault.records]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(times_f, tvcs_f, label='TVC', color='blue')
axes[0].plot(times_f, tambs_f, label='TAMB', color='orange', alpha=0.6)
axes[0].axhline(2, color='cyan', linestyle='--', alpha=0.5)
axes[0].axhline(8, color='red', linestyle='--', alpha=0.5)
fault_start_time = start + dt.timedelta(hours=8)
fault_end_time = start + dt.timedelta(hours=24*10)
axes[0].axvspan(fault_start_time, fault_end_time, alpha=0.15, color='red', label='Compressor failure')
axes[0].set_ylabel('TVC (°C)')
axes[0].set_title('Compressor failure — TVC excursion')
axes[0].legend()

axes[1].bar(times_f, cmprs_f, width=dt.timedelta(minutes=8), color='green', alpha=0.7)
axes[1].axvspan(fault_start_time, fault_end_time, alpha=0.15, color='red')
axes[1].set_ylabel('Compressor runtime (s)')
axes[1].set_xlabel('Time')

plt.tight_layout()
plt.show()

## 4. Door behavior presets

`EventConfig` provides named presets calibrated from fleet data (1,225 fridges, Jan 2021 – Dec 2022). Compare well-managed vs. inattentive door use patterns and their thermal impact.

In [ ]:
from utils.simulator.config import EventConfig

presets = {
    'bestpractice': EventConfig.bestpractice(),
    'normal': EventConfig.normal(),
    'few_but_long': EventConfig.few_but_long(),
    'frequent_short': EventConfig.frequent_short(),
    'busy_facility': EventConfig.busy_facility(),
}

fig, axes = plt.subplots(len(presets), 2, figsize=(14, 3 * len(presets)), sharex='col')
start = dt.datetime(2024, 6, 15, 0, 0, 0)
sample_days = 3

for row, (name, ev) in enumerate(presets.items()):
    cfg = default_config(power_type="mains", latitude=12.0)
    cfg.sample_interval = 600  # 10-minute samples
    cfg.events = ev
    batch_size = int(86400 / cfg.sample_interval * sample_days)
    rs = SimulatedRecordSet.generate(cfg, batch_size=batch_size, start_time=start, interval=cfg.sample_interval)

    times = [r['ABST'] for r in rs.records]
    tvcs = [r['TVC'] for r in rs.records]
    dorvs = [r['DORV'] for r in rs.records]
    total_opens = sum(1 for d in dorvs if d > 0)
    total_door_s = sum(dorvs)

    axes[row, 0].plot(times, tvcs, color='blue', linewidth=0.8)
    axes[row, 0].axhline(2, color='cyan', linestyle='--', alpha=0.4)
    axes[row, 0].axhline(8, color='red', linestyle='--', alpha=0.4)
    axes[row, 0].set_ylabel('TVC (°C)')
    axes[row, 0].set_title(f'{name} — TVC')

    axes[row, 1].bar(times, dorvs, width=dt.timedelta(minutes=8), color='magenta', alpha=0.7)
    axes[row, 1].set_ylabel('Door (s)')
    axes[row, 1].set_title(f'{name} — {total_opens} opens, {total_door_s:.0f}s total over {sample_days}d')

axes[-1, 0].set_xlabel('Time')
axes[-1, 1].set_xlabel('Time')
plt.tight_layout()
plt.show()

## 5. WHO alarm rules (HEAT / FRZE)

Alarms now follow WHO PQS continuous-excursion rules:
- **HEAT**: triggers after **10 continuous hours** with TVC > +8 °C
- **FRZE**: triggers after **60 continuous minutes** with TVC ≤ −0.5 °C

Below we simulate a prolonged power outage (HEAT excursion) and a thermostat malfunction driving freeze exposure.

In [ ]:
from utils.simulator.config import FaultType

start = dt.datetime(2024, 6, 15, 0, 0, 0)
sample_days = 14
interval = 600  # 10-minute samples
batch_size = int(86400 / interval * sample_days)

# --- HEAT alarm: prolonged power outage at equatorial site ---
# Icebank holds for several days, then TVC rises past +8°C.
# HEAT triggers after 10 continuous hours above +8°C.
heat_cfg = default_config(power_type="mains", latitude=2.0)
heat_cfg.sample_interval = interval
heat_cfg.fault.fault_type = FaultType.POWER_OUTAGE
heat_cfg.fault.fault_start_offset_s = 6 * 3600
heat_cfg.fault.fault_duration_s = 14 * 86400

rs_heat = SimulatedRecordSet.generate(heat_cfg, batch_size=batch_size, start_time=start, interval=interval)

# --- FRZE alarm: thermostat malfunction (stuck-low setpoint) ---
# Compressor overcools the chamber below -0.5°C.
# FRZE triggers after 60 continuous minutes at or below -0.5°C.
frze_cfg = default_config(power_type="mains", latitude=12.0)
frze_cfg.sample_interval = interval
frze_cfg.thermal.icebank_capacity_j = 0.0
frze_cfg.thermal.compressor_targets_icebank = False
frze_cfg.thermal.Q_compressor = 200.0
frze_cfg.thermal.T_setpoint_low = -8.0
frze_cfg.thermal.T_setpoint_high = -5.0

rs_frze = SimulatedRecordSet.generate(frze_cfg, batch_size=batch_size, start_time=start, interval=interval)

# --- Plot both ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, rs, label, threshold, color, alarm_code, fault_desc in [
    (axes[0], rs_heat, 'Power outage → HEAT alarm', 8.0, 'red', 'HEAT',
     'Power outage (6h–end)'),
    (axes[1], rs_frze, 'Thermostat malfunction → FRZE alarm', -0.5, 'blue', 'FRZE',
     'Setpoint stuck at -8°C'),
]:
    times = [r['ABST'] for r in rs.records]
    tvcs = [r['TVC'] for r in rs.records]
    alrms = [r['ALRM'] for r in rs.records]

    ax.plot(times, tvcs, color='steelblue', linewidth=0.8, label='TVC')
    ax.axhline(threshold, color=color, linestyle='--', alpha=0.5,
               label=f'Threshold ({threshold}°C)')

    # Shade alarm-active intervals (alarm codes may be combined, e.g. "HEAT POWR")
    alarm_times = [t for t, a in zip(times, alrms) if a and alarm_code in a]
    if alarm_times:
        ax.axvspan(alarm_times[0], alarm_times[-1], alpha=0.12, color=color,
                   label=f'{alarm_code} alarm active')

    ax.set_ylabel('TVC (°C)')
    ax.set_title(label)
    ax.legend(loc='upper right', fontsize=9)

    # Annotate first alarm timestamp
    if alarm_times:
        first_alarm = alarm_times[0]
        tvc_at_alarm = tvcs[times.index(first_alarm)]
        ax.annotate(f'{alarm_code} triggered',
                    xy=(first_alarm, tvc_at_alarm),
                    xytext=(first_alarm + dt.timedelta(hours=18), tvc_at_alarm + 2),
                    arrowprops=dict(arrowstyle='->', color=color),
                    fontsize=9, color=color)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

# Print alarm summary
for name, rs in [('HEAT scenario', rs_heat), ('FRZE scenario', rs_frze)]:
    alrm_records = [(r['ABST'], r['ALRM']) for r in rs.records if r['ALRM'] is not None]
    if alrm_records:
        print(f"{name}: {len(alrm_records)} alarm records, "
              f"first at {alrm_records[0][0]} ({alrm_records[0][1]})")
    else:
        print(f"{name}: no alarms triggered")

## 6. High-level: BaseRtmDevice

The `BaseRtmDevice` class wraps the simulator with device metadata and Pydantic schema validation. This is what `locustfile.py` uses.

In [ ]:
# Create a virtual EMS device
config = MonitoringDeviceConfig(type='ems', upload_interval=3600, sample_interval=900)
device = BaseRtmDevice(config)
print(device)
print(f"Power source: {device.powersource}")
print(f"Facility: {device.config.facility.facility_name}")
print(f"Batch size per report: {device.config.batch_size}")

# Generate 3 sequential reports (simulating 3 upload cycles)
t = dt.datetime(2024, 6, 15, 6, 0, 0)
for i in range(3):
    report = device.create_report(report_time=t)
    print(f"\nReport {i+1} at {t}:")
    print(f"  Type: {type(report).__name__}")
    print(f"  Records: {len(report.records)}")
    for rec in report.records:
        print(f"    {rec.ABST} | TVC={rec.TVC:5.1f} | TAMB={rec.TAMB:5.1f}")
    t += dt.timedelta(hours=1)

## 7. JSON serialization (CCDX format)

Reports serialize to the CCDX transfer schema, ready for POST to OpenFn.

In [ ]:
import json
from utils.schemas import TransferMetadata, EmsTransfer
from utils.generator import transfer_metadata

config = MonitoringDeviceConfig(type='ems', upload_interval=3600, sample_interval=900)
device = BaseRtmDevice(config)
report = device.create_report(report_time=dt.datetime(2024, 6, 15, 12, 0, 0))

md = transfer_metadata(type='ems')
transfer = EmsTransfer(data=[report], meta=TransferMetadata(**md))
body = transfer.model_dump(mode='json', exclude_unset=True)

print(json.dumps(body, indent=2, default=str)[:2000])
print("...")